# Notebook to demonstrate use of FOMO's JPLSBDBQuery class to query the JPL Small Body Database (SBDB)

(Also not coincidentally also used to figure out which comets might be visible and work out a time budget)

**Last updated:** 2026-03-24 23:40 UTC

**Credit/Blame:** Tim Lister

In [ ]:
import os
import sys
import django

# Add your Django project path to the Python path
# Option: adjust the path as necessary for your project structure
sys.path.insert(0, os.path.join(os.getenv('HOME'), 'git', 'fomo'))

# Set the DJANGO_SETTINGS_MODULE environment variable
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'fomo.settings')

# Initialize Django
django.setup()

print('✅ Django successfully configured!')

In [ ]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from astropy.time import Time
from astropy.table import QTable, Column
from astropy import units as u
from astropy.utils.exceptions import AstropyUserWarning
from astroquery.jplhorizons import Horizons
import warnings
from erfa import ErfaWarning
import requests
from socket import timeout
import numpy as np
from solsys_code.views import JPLSBDBQuery

In [ ]:
kp_start = datetime(2026, 8, 1)
kp_end = datetime(2029, 8, 1)
kp_midpoint = kp_start + (kp_end - kp_start) / 2
time_fmt = '%Y-%m-%d'
print(f'Considering {kp_start.strftime(time_fmt)} -> {kp_end.strftime(time_fmt)} (midpoint= {kp_midpoint})')

## Query JPL Small Body DataBase for comets with perihelion dates within +/-3 years of midpoint

Query for all possible comet orbit classes according to the definition in the [Available Orbit Classes](https://ssd-api.jpl.nasa.gov/doc/sbdb_filter.html#available-sbdb-orbit-classes):
| Code | Name | Kind | Description |
|------|------|------|-------------|
| ETc | Encke-type Comet | comet | Encke-type comet, as defined by Levison and Duncan (Tj > 3; a < aJ) |
| JFc | Jupiter-family Comet | comet | Jupiter-family comet, as defined by Levison and Duncan (2 < Tj < 3) |
| JFC | Jupiter-family Comet* | comet | Jupiter-family comet, classical definition (P < 20 y) |
| CTc | Chiron-type Comet | comet | Chiron-type comet, as defined by Levison and Duncan (Tj > 3; a > aJ) |
| HTC | Halley-type Comet* | comet | Halley-type comet, classical definition (20 y < P < 200 y) |
| PAR | Parabolic Comet | comet | Comets on parabolic orbits (e = 1.0) |
| HYP | Hyperbolic Comet | comet | Comets on hyperbolic orbits (e > 1.0) |
| COM | Comet | comet | Comet orbit not matching any defined orbit class |

_(Note that the * on JFC and HTC names denotes the classical definition for that orbit class, as flagged in the page's introductory note. The two JFc/JFC entries are distinct codes — case-sensitive, as the API docs warn — representing the Levison & Duncan definition vs the classical period-based definition respectively.)_

We will select the classical definitions so the full list is `JFC,HTC,PAR,HYP,COM`

In [ ]:
comet_classes = 'JFC,HTC,PAR,HYP,COM'
# Convert the midpoint time into start and end times in JD in the TDB timescale that JPL SBDB represents perihelion times with
# This is wrapped to catch warnings since we can't predict leapseconds in the UTC timescale that far into the future
with warnings.catch_warnings():
    # Catch possible dubious year warnings from erfa
    warnings.filterwarnings('ignore', category=ErfaWarning)
    result = kp_start.replace(year=kp_start.year - 3)
    tp_start = Time(result, scale='utc')
    result = kp_end.replace(year=kp_start.year + 3)
    tp_end = Time(result, scale='utc')
    comet_constraints = [f'{tp_start.tdb.jd:.2f} <= tp <= {tp_end.tdb.jd:.2f}']

jpl = JPLSBDBQuery(orbit_class=comet_classes, orbital_constraints=comet_constraints)

In [ ]:
print(jpl.__dict__)

In [ ]:
results = jpl.run_query()

In [ ]:
# Doublecheck we got the expected signature before proceeding
assert results['signature'] == {'version': '1.0', 'source': 'NASA/JPL SBDB (Small-Body DataBase) Query API'}

In [ ]:
print(f"Retrieved {results['count']} potential comets")
print(results['data'][0])

### Parse retrieved results into an AstroPy `QTable` and sort by perihelion date (`tp`)

In [ ]:
table = jpl.parse_results(results)
table.sort('tp')

In [ ]:
table.pprint(max_width=-1)

### Generate ephemeris for each comet

Compute a coarse (1d step size) first to see if it's not some crazy magnitude. Then refine for actual visibility. We also need a helper routine ported from NEOexchange in order to resolve names into unique Horizons SPK ids in some cases (almost always short-period comets with many element sets)

In [ ]:
def determine_horizons_id(lines, now=None):
    """Attempts to determine the HORIZONS id of a target body that has multiple
    possibilities. The passed [lines] (from the .args attribute of the exception)
    are searched for the HORIZONS id (column 1) whose 'epoch year' (column 2)
    which is closest to [now] (a passed-in datetime or defaulting to datetime.utcnow()"""

    now = now or datetime.utcnow()
    timespan = timedelta.max
    horizons_id = None
    for line in lines:
        chunks = line.split()
        if len(chunks) >= 5 and chunks[0].isdigit() is True and chunks[1].isdigit() is True:
            try:
                epoch_yr = datetime.strptime(chunks[1], '%Y')
                if abs(now - epoch_yr) <= timespan:
                    # New closer match to "now"
                    horizons_id = int(chunks[0])
                    timespan = now - epoch_yr
            except ValueError:
                print('Unable to parse year of epoch from', line)
    return horizons_id

In [ ]:
def convert_horizons_table(ephem, include_moon=False):
    """Modifies a passed table <ephem> from the `astroquery.jplhorizons.ephemerides()
    to add a 'datetime' column, rate columns and remove the Civil and Nautical twilight-flagged rows.
    The modified Astropy Table is returned"""

    twilight_mask = (ephem['solar_presence'] != 'C') & (
        ephem['solar_presence'] != 'N'
    )  # &  (ephem['solar_presence'] != 'A')
    new_ephem = ephem[twilight_mask]
    dates = Time([datetime.strptime(d, '%Y-%b-%d %H:%M') for d in new_ephem['datetime_str']])
    if 'datetime' not in new_ephem.colnames:
        new_ephem.add_column(dates, name='datetime')
    # Convert units of RA/Dec rate from arcsec/hr to arcsec/min and compute
    # mean rate
    new_ephem['RA_rate'].convert_unit_to('arcsec/min')
    new_ephem['DEC_rate'].convert_unit_to('arcsec/min')
    rate_units = new_ephem['DEC_rate'].unit
    mean_rate = np.sqrt(new_ephem['RA_rate'] ** 2 + new_ephem['DEC_rate'] ** 2)
    mean_rate.unit = rate_units
    new_ephem.add_column(mean_rate, name='mean_rate')

    return new_ephem

In [ ]:
def horizons_ephem(
    obj_name,
    start,
    end,
    site_code,
    ephem_step_size='1h',
    airmass_limit=2,
    should_skip_daylight=False,
    ha_limit=0,
    include_moon=False,
):
    horizons_quantities = '1,3,4,9,19,20,23,24,38,42,33,47'

    eph = Horizons(
        id=obj_name,
        id_type='smallbody',
        epochs={
            'start': start.strftime('%Y-%m-%d %H:%M'),
            'stop': end.strftime('%Y-%m-%d %H:%M'),
            'step': ephem_step_size,
        },
        location=site_code,
    )
    ephem = []
    try:
        ephem = eph.ephemerides(
            quantities=horizons_quantities,
            skip_daylight=should_skip_daylight,
            airmass_lessthan=airmass_limit,
            max_hour_angle=ha_limit,
            cache=True,
        )
        ephem = convert_horizons_table(ephem, include_moon)
    except ConnectionError as e:
        print('Unable to connect to HORIZONS')
    except requests.exceptions.ConnectionError as e:
        print('Unable to connect to HORIZONS')
    except timeout as sock_e:
        print(f'HORIZONS retrieval failed with socket Error {sock_e.errno}: {sock_e.strerror}')
    except ValueError as e:
        if e.args and len(e.args) > 0:
            if 'No ephemeris meets criteria' in e.args[0]:
                print('No visibility for object')
            else:
                print('Ambiguous object, trying to determine HORIZONS id')

                choices = e.args[0].split('\n')
                horizons_id = determine_horizons_id(choices)
                if horizons_id:
                    try:
                        eph = Horizons(
                            id=horizons_id,
                            epochs={
                                'start': start.strftime('%Y-%m-%d %H:%M:%S'),
                                'stop': end.strftime('%Y-%m-%d %H:%M:%S'),
                                'step': ephem_step_size,
                            },
                            location=site_code,
                        )
                        ephem = eph.ephemerides(
                            quantities=horizons_quantities,
                            skip_daylight=should_skip_daylight,
                            airmass_lessthan=airmass_limit,
                            max_hour_angle=ha_limit,
                        )
                        ephem = convert_horizons_table(ephem, include_moon)
                    except ValueError as e:
                        print('Error querying HORIZONS. Error message: {}'.format(e))
                else:
                    print('Unable to determine the HORIZONS id')
        else:
            print('Error querying HORIZONS. Error message: {}'.format(e))
    return ephem

In [ ]:
# Define HORIZONS ephemeris parameters and make output directory (if it does not exist)
ephem_step_size = '1d'
site_code = '-1'
# Output path to save ephemerides into
output_path = os.path.join(os.getcwd(), 'data', '')
if os.path.exists(output_path) is False:
    os.makedirs(output_path)

In [ ]:
should_skip_daylight = False
airmass_limit = 99
ha_limit = 0
for obj_name in table['pdes']:
    print(f'Generating ephemeris for {obj_name}', end='')
    ephem_filename = f"{obj_name.replace(' ', '').replace('/', '_')}_ephem_{kp_start.strftime('%Y%m%d')}-{kp_end.strftime('%Y%m%d')}-{ephem_step_size}.ecsv"
    ephem_filename = os.path.join(output_path, ephem_filename)
    if os.path.exists(ephem_filename) is True:
        print(' Ephemeris file already exists')
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=AstropyUserWarning)
            ephem = QTable.read(ephem_filename, format='ascii.ecsv')
    else:
        ephem = horizons_ephem(
            obj_name, kp_start, kp_end, site_code, ephem_step_size, airmass_limit, should_skip_daylight, ha_limit
        )
        ephem.write(ephem_filename, format='ascii.ecsv', overwrite=True)
        print(f'  {len(ephem)} lines returned and written to {os.path.basename(ephem_filename)}')

In [ ]:
print(ephem)

In [ ]:
print(ephem.columns)

In [ ]:
%matplotlib
import matplotlib.pyplot as plt

In [ ]:
plt.plot(ephem['datetime_jd'], ephem['Tmag'], 'b--')
ax = plt.gca()
ax.invert_yaxis()
plt.title('Magnitude for ' + obj_name)

### Check for valid magnitudes
Working on V<18 for now

In [ ]:
valid_objects = []
min_days = 3
mag_limit = 18
for prefix, obj_name in table[('prefix', 'pdes')]:
    valid = False
    ephem_filename = f"{obj_name.replace(' ', '').replace('/', '_')}_ephem_{kp_start.strftime('%Y%m%d')}-{kp_end.strftime('%Y%m%d')}-{ephem_step_size}.ecsv"
    ephem_filename = os.path.join(output_path, ephem_filename)
    if os.path.exists(ephem_filename) is True:
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=AstropyUserWarning)
            ephem = QTable.read(ephem_filename, format='ascii.ecsv')
        mag_mask = ephem['Tmag'] <= mag_limit * u.mag
        if len(ephem[mag_mask]) >= min_days:
            valid = True
            valid_objects.append(str(obj_name))
        if prefix != '':
            prefix += '/'
        else:
            prefix = '  '
        row = ephem['Tmag'].argmin()
        min_Vmag = ephem['Tmag'][row]
        if row == 0:
            time_of_min_Vmag = 'KP start'
        else:
            time_of_min_Vmag = ephem['datetime_str'][row]
        print(
            f'{prefix}{obj_name:<9s}: {len(ephem[mag_mask]):4d} days visible (Brightest V={min_Vmag:4.1f} at {time_of_min_Vmag})'
        )

### Remove dead/dying/disintegrating comets first

In [ ]:
dead_comets = set(['2025 K1', '2024 E1'])
valid_objects[:] = [x for x in valid_objects if x not in dead_comets]

In [ ]:
print(f'{len(valid_objects)} potential comet targets')

In [ ]:
print(valid_objects)

### Create more detailed ephemeris to refine target selection


In [ ]:
def get_semester(dt: datetime) -> str:
    """Return the semester name for a given datetime.

    Semesters are 6 months long, alternating A (Feb 1) and B (Aug 1).
    e.g. 2026B runs 2026-08-01 to 2027-01-31
         2027A runs 2027-02-01 to 2027-07-31
    """
    # Semester boundaries are Feb 1 and Aug 1
    if dt.month < 2:
        # Jan: still in previous year's B semester
        return f'{dt.year - 1}B'
    elif dt.month < 8:
        # Feb–Jul: A semester
        return f'{dt.year}A'
    else:
        # Aug–Dec: B semester
        return f'{dt.year}B'


def get_all_semesters(start: datetime, n: int = 6) -> list[dict]:
    """Return a list of n semester dicts with name, start, and end dates."""
    semesters = []
    current = start
    for _ in range(n):
        name = get_semester(current)
        end = current + relativedelta(months=6) - relativedelta(days=1)
        semesters.append({'name': name, 'start': current, 'end': end})
        current += relativedelta(months=6)
    return semesters

In [ ]:
# ── Generate semesters for Key Project
start = datetime(2026, 8, 1)
semesters = get_all_semesters(start, n=6)

In [ ]:
comet_semesters = {}
prefix = 'C/'
for obj_name in valid_objects:
    ephem_filename = f"{obj_name.replace(' ', '').replace('/', '_')}_ephem_{kp_start.strftime('%Y%m%d')}-{kp_end.strftime('%Y%m%d')}-{ephem_step_size}.ecsv"
    ephem_filename = os.path.join(output_path, ephem_filename)
    if os.path.exists(ephem_filename) is True:
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore', category=AstropyUserWarning)
            ephem = QTable.read(ephem_filename, format='ascii.ecsv')
        mag_mask = ephem['Tmag'] <= mag_limit * u.mag
        row = ephem['Tmag'].argmin()
        min_Vmag = ephem['Tmag'][row]
        time_of_min_Vmag = datetime.strptime(ephem['datetime_str'][row], '%Y-%b-%d %H:%M')
        semester = get_semester(time_of_min_Vmag)
        if semester not in comet_semesters:
            comet_semesters[semester] = []
        comet_semesters[semester].append(obj_name)
        print(
            f'{prefix}{obj_name:<9s}: {len(ephem[mag_mask]):4d} days visible (Brightest V={min_Vmag:4.1f} at {time_of_min_Vmag.strftime(time_fmt)} {semester})'
        )

print('\n\n Number of objects per semester')
print('-' * 43)
for s in semesters:
    print(f"{s['name']}  {s['start'].date()} → {s['end'].date()} #objects= {len(comet_semesters.get(s['name'], []))}")

#### Create more detailed ephemerides for objects in 2026B

In [ ]:
semester = '2026B'
semester_start = semesters[0]['start']
semester_end = semesters[0]['end']
print(f'Generating ephemerides for {semester} ({semester_start} -> {semester_end})')
ephem_step_size = '15m'
should_skip_daylight = False
airmass_limit = 2
ha_limit = 4.7
site_codes = ['Z24', 'K92']
for obj_name in comet_semesters[semester]:
    print(obj_name)
    for site_code in site_codes:
        print(f'Generating ephemeris for {obj_name} for {site_code}')
        ephem_filename = f"{obj_name.replace(' ', '').replace('/', '_')}_ephem_{semester_start.strftime('%Y%m%d')}-{semester_end.strftime('%Y%m%d')}-{ephem_step_size}_{site_code}.ecsv"
        ephem_filename = os.path.join(output_path, ephem_filename)
        if os.path.exists(ephem_filename) is True:
            print(' Ephemeris file already exists')
            with warnings.catch_warnings():
                warnings.filterwarnings('ignore', category=AstropyUserWarning)
                ephem = QTable.read(ephem_filename, format='ascii.ecsv')
        else:
            ephem = horizons_ephem(
                obj_name,
                semester_start,
                semester_end,
                site_code,
                ephem_step_size,
                airmass_limit,
                should_skip_daylight,
                ha_limit,
            )
            if len(ephem) > 0:
                ephem.write(ephem_filename, format='ascii.ecsv', delimiter=',', overwrite=True)
            print(f'  {len(ephem)} lines returned and written to {os.path.basename(ephem_filename)}')

## Figure out time usage
This is assuming a sparse sampling of 1hr per night for  7 days

In [ ]:
# Overhead values for 1m + Sinistro (most will be same for Sophia except readout time, which is currently unknown)
t_setup = 90 * u.s
t_filter = 18 * u.s
t_readout_ff = 28 * u.s  # (Full Frame)
t_readout_2k = 4 * u.s  # (Central 2k x 2k, bin2)

In [ ]:
t_visit = 1 * u.hr
num_sites_per_night = 3
num_nights = 7
total_time_per_object = t_visit * num_sites_per_night * num_nights
total_time = total_time_per_object * len(valid_objects)
print(f'Time needed per object= {total_time_per_object:.1f}\nTotal time needed= {total_time:.1f}')